## Setup

In [1]:
# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
sns.set(style='whitegrid')
%matplotlib inline

## Data

### Loading

In [2]:
#I have downloaded the data set from https://www.kaggle.com/datasets/blastchar/telco-customer-churn
#I have renamed the file telco_churn.csv and set in the proper directory
df = pd.read_csv('../data/raw/telco_churn.csv')
#Let us look at the head of the dataframe

### Cleaning

In [3]:
#I can convert TotalCharges to a numeric value
df["TotalCharges"] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("The type is now",df['TotalCharges'].dtype)
if df['TotalCharges'].isna().any():
    print("NaNs detected in TotalCharges after transform")

#Now, I want to give fields a correct type. Let me start with the binary columns
# For gender
if df['gender'].dtype == 'str':
    df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})

# For other binary columns
binary_columns=['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_columns:
    if df[col].dtype == 'str':
        df[col] = df[col].map({'Yes': 1, 'No': 0})

#Now transform MultipleLines to binary
df['MultipleLines'] = (df['MultipleLines'] == 'Yes').astype(int)
#This is safe because 'No phone service' carries no additional info beyond PhoneService.

#I want to turn Internet service into binary
# Step 1: Preserve original values only if not already preserved
if 'InternetService_Category' not in df.columns:
    df['InternetService_Category'] = df['InternetService']
# Step 2: Create a binary column safely
df['InternetService'] = (df['InternetService'] != 'No').astype(int)

internet_services = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

for col in internet_services:
    # Only transform if the sanity check passes
    sanity_passed = (df.loc[df[col] == 'No internet service', 'InternetService'] == 0).all()
    
    if sanity_passed:
        # Idempotent binary conversion: 1 if 'Yes', 0 otherwise
        df[col] = (df[col] == 'Yes').astype(int)
    else:
        raise ValueError(f"Sanity check failed for {col}, not transforming")
    
#I one-hot encode before feature engineering
categorical_columns=['Contract','PaymentMethod','InternetService_Category']
for col in categorical_columns:
    # One-hot encode and convert to int
    dummies = pd.get_dummies(df[col], prefix=col).astype(int)
    df = pd.concat([df, dummies], axis=1)

The type is now float64
NaNs detected in TotalCharges after transform


### Feature Engineering

In [4]:
#Feature Engineering

# 1. Tenure Group
if 'tenure_group' not in df.columns:
    df['tenure_group'] = pd.cut(
        df['tenure'],
        bins=[0, 12, 24, 48, 72],
        labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr']
    )

#2. New costumer flag
if 'is_new_customer' not in df.columns:
    df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

# 3. Long-Term Contract Flag
if 'is_long_term' not in df.columns:
    df['is_long_term'] = (
        df['Contract_One year'] + df['Contract_Two year']
    ).astype(int)

# 4. Auto-Pay Flag
if 'auto_pay_flag' not in df.columns:
    df['auto_pay_flag'] = (
        df['PaymentMethod_Bank transfer (automatic)'] +
        df['PaymentMethod_Credit card (automatic)']
    ).astype(int)

# 5. Number of services
if 'num_services' not in df.columns:
    service_cols = [
        'OnlineSecurity',
        'OnlineBackup',
        'DeviceProtection',
        'TechSupport',
        'StreamingTV',
        'StreamingMovies'
    ]
    df['num_services'] = df[service_cols].sum(axis=1)

# 6. Average Revenue Per Month
if 'avg_revenue' not in df.columns:
    df['avg_revenue'] = df['TotalCharges'] / (df['tenure'] + 1)

# 7. High Monthly Charges Flag
if 'high_monthly_flag' not in df.columns:
    median_charge = df['MonthlyCharges'].median()
    df['high_monthly_flag'] = (df['MonthlyCharges'] > median_charge).astype(int)

# 8. Family Flag
if 'family_flag' not in df.columns:
    df['family_flag'] = (
        (df['Partner'] + df['Dependents']) > 0
    ).astype(int)

# 9. Fiber Optic Flag
if 'fiber_flag' not in df.columns:
    df['fiber_flag'] = df['InternetService_Category_Fiber optic']

# 10. Electronic Check Flag
if 'electronic_check_flag' not in df.columns:
    df['electronic_check_flag'] = df['PaymentMethod_Electronic check']

#Interaction features

# 1. Interaction: New Customer × Electronic Check
if 'new_echeck_interaction' not in df.columns:
    df['new_echeck_interaction'] = (
        df['is_new_customer'] * df['electronic_check_flag']
    )

# 2. Interaction: Fiber × High Monthly Charge
if 'fiber_highcharge_interaction' not in df.columns:
    df['fiber_highcharge_interaction'] = (
        df['fiber_flag'] * df['high_monthly_flag']
    )

# 3. Interaction: Long-Term Contract × Number of Services
if 'loyal_engaged_interaction' not in df.columns:
    df['loyal_engaged_interaction'] = (
        df['is_long_term'] * df['num_services']
    )

## Preprocessing

### Target and Features Matrices

In [5]:
# 1. Target
y = df['Churn']

# 2. Feature selection
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_revenue']
engineered_features = [
    'is_new_customer', 'is_long_term', 'auto_pay_flag', 'num_services',
    'high_monthly_flag', 'family_flag', 'fiber_flag', 'electronic_check_flag',
    'new_echeck_interaction', 'fiber_highcharge_interaction', 'loyal_engaged_interaction'
]
categorical_features = ['Contract', 'PaymentMethod', 'InternetService_Category', 'tenure_group']

# 3. X matrix for all models
X = df[numeric_features + engineered_features + categorical_features]

# At this point X is just raw data; preprocessing will come later


### Train / Test Split

In [6]:
from sklearn.model_selection import train_test_split

# Split once for all models (stratify ensures class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

### Preprocessing

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# -------------------------------
# Reusable pipeline steps
# -------------------------------
median_imputer = SimpleImputer(strategy='median')
most_frequent_imputer = SimpleImputer(strategy='most_frequent')
missing_cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
scaler = StandardScaler()
ohe_drop_first = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ohe_all = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# -------------------------------
# 1. Linear model preprocessing
# -------------------------------
num_pipeline_linear = Pipeline([
    ('imputer', median_imputer),
    ('scaler', scaler)
])

cat_pipeline_linear = Pipeline([
    ('imputer', missing_cat_imputer),
    ('encoder', ohe_drop_first)
])

linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_linear, numeric_features + engineered_features),
        ('cat', cat_pipeline_linear, categorical_features)
    ]
)

# -------------------------------
# 2. Tree-based preprocessing
# -------------------------------
num_pipeline_tree = Pipeline([
    ('imputer', median_imputer)
])

cat_pipeline_tree = Pipeline([
    ('imputer', most_frequent_imputer),
    ('encoder', ohe_all)
])

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_tree, numeric_features + engineered_features),
        ('cat', cat_pipeline_tree, categorical_features)
    ]
)

# 3. Fit and transform X matrices
X_train_linear = linear_preprocessor.fit_transform(X_train)
X_test_linear  = linear_preprocessor.transform(X_test)

X_train_tree   = tree_preprocessor.fit_transform(X_train)
X_test_tree    = tree_preprocessor.transform(X_test)

# Ready for modeling:
# LogisticRegression → X_train_linear / X_test_linear
# RandomForest / GradientBoosting → X_train_tree / X_test_tree


## Modelling

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score


# Instantiate baseline logistic regression
logreg = LogisticRegression(random_state=42, max_iter=1000)

# Fit on preprocessed linear training data
logreg.fit(X_train_linear, y_train)

# Predict
y_pred_logreg = logreg.predict(X_test_linear)

y_proba_logreg = logreg.predict_proba(X_test_linear)[:, 1]
# Evaluate
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_logreg))




# Instantiate baseline random forest
rf = RandomForestClassifier(random_state=42, n_estimators=100)

# Fit on tree-preprocessed data
rf.fit(X_train_tree, y_train)

# Predict
y_pred_rf = rf.predict(X_test_tree)
y_proba_rf = rf.predict_proba(X_test_tree)[:, 1]

# Evaluate
print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))



# Instantiate baseline gradient boosting
gb = GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1)

# Fit on tree-preprocessed data
gb.fit(X_train_tree, y_train)

# Predict
y_pred_gb = gb.predict(X_test_tree)
y_proba_gb = gb.predict_proba(X_test_tree)[:, 1]

# Evaluate
print("Gradient Boosting")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))

Logistic Regression
Accuracy: 0.7899219304471257
Random Forest
Accuracy: 0.7757274662881476
Gradient Boosting
Accuracy: 0.794889992902768


## Evaluation

In [ ]:
# Model Comparison

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

# Collect metrics in a dictionary
metrics = {
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_logreg),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_logreg),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_logreg),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb)
    ]
}

# Create a DataFrame for clean comparison
comparison_df = pd.DataFrame(metrics)
comparison_df

,Model,Accuracy,ROC-AUC,Precision,Recall,F1 Score
0,Logistic Regression,0.789922,0.841700,0.637324,0.483957,0.550152
1,Random Forest,0.775727,0.809818,0.592357,0.497326,0.540698
2,Gradient Boosting,0.794890,0.838526,0.639344,0.521390,0.574374
